In [26]:
from sklearn import metrics
from neural_network import normalize, train_neural_network, DATA_HEADERS
import pandas as pd
import numpy as np

In [16]:
non_bag_url = "https://raw.githubusercontent.com/reganq/csc311-project/refs/heads/main/cleaned_data.csv"
non_bag_df = pd.read_csv(non_bag_url)
bag_url = "../cleaned_data_bag.csv"
bag_df = pd.read_csv(bag_url)

In [17]:
df = non_bag_df.copy()

In [18]:
# split the data
train_df = df[df['is_train'] == True]
val_df = df[df['is_train'] == False]

# split off the labels - they cannot be one-hot encoded for logreg. Instead, we have 3 targets (0, 1, 2)
t_train = np.argmax(np.stack([
    (train_df['painting'] == 'The Persistence of Memory').astype(np.int8),
    (train_df['painting'] == 'The Starry Night').astype(np.int8),
    (train_df['painting'] == 'The Water Lily Pond').astype(np.int8)
]), axis=0)

t_val = np.argmax(np.stack([
    (val_df['painting'] == 'The Persistence of Memory').astype(np.int8),
    (val_df['painting'] == 'The Starry Night').astype(np.int8),
    (val_df['painting'] == 'The Water Lily Pond').astype(np.int8)
]), axis=0)

# headers = DATA_HEADERS
headers = df.columns.values.tolist()
headers.remove('unique_id')
headers.remove('painting')
headers.remove('is_train')

X_train = np.array(train_df.get(headers))
X_train = X_train.astype(np.int8) # get rid of those pesky floats
X_val = np.array(val_df.get(headers))
X_val = X_val.astype(np.int8)

In [19]:
# normalize the (non-BOW) data
X_train[:20] = normalize(X_train[:20])
X_val[:20] = normalize(X_val[:20])

c:\Users\ecorb\OneDrive\Documents\school\2025-26\CSC311\csc311-project\neural_network\neural_network.py:80: RuntimeWarning: invalid value encountered in divide
  return (X - mean) / std
C:\Users\ecorb\AppData\Local\Temp\ipykernel_62832\1208814450.py:2: RuntimeWarning: invalid value encountered in cast
  X_train[:20] = normalize(X_train[:20])
C:\Users\ecorb\AppData\Local\Temp\ipykernel_62832\1208814450.py:3: RuntimeWarning: invalid value encountered in cast
  X_val[:20] = normalize(X_val[:20])


In [20]:
def build_all_models(alpha,
                     activation,
                     batch_size,
                     hidden_layer_sizes,
                     X_train=X_train,
                     t_train=t_train,
                     X_valid=X_val,
                     t_valid=t_val):
    """
    Returns a dictionary, `out`, whose keys are the the hyperparameter choices, and whose values are
    the training and validation accuracies (via the `score()` method).
    Arguments:
        - alpha: Regularization strength.
        - activation: Which activation function to use.
        - batch_size: Size of the mini-batches for stochastic optimizers.
        - hidden_layer_sizes: Tuple of integers. The i-th element represents the number of neurons in the i-th hidden layer.
    """
    out = {}

    for a in alpha:
        for act in activation:
            for b in batch_size:
                for h in hidden_layer_sizes:
                    out[(a, act, b, h)] = {}
                    # Create a neural network model based on the given hyperparameters and fit it to the data
                    model = train_neural_network(X_train, t_train, alpha=a, activation=act, batch_size=b, hidden_layer_sizes=h)
                    
                    # store the validation and training scores in the `out` dictionary
                    out[(a, act, b, h)]['val'] = model.score(X_valid, t_valid)
                    out[(a, act, b, h)]['train'] = model.score(X_train, t_train)
    return out

In [21]:
first_model = train_neural_network(X_train, t_train, alpha=0.0001, activation='relu', batch_size=32, hidden_layer_sizes=(100,))
print("First model training accuracy:", first_model.score(X_train, t_train))
print("First model validation accuracy:", first_model.score(X_val, t_val))

First model training accuracy: 0.9464285714285714
First model validation accuracy: 0.8053097345132744


C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [22]:
# Hyperparameters values
alpha = [0.0001, 0.001, 0.01]
activation = ['relu', 'tanh']
batch_size = [16, 32]
hidden_layer_sizes = [(50,), (100,), (50, 50)]

res = build_all_models(alpha=alpha, activation=activation, batch_size=batch_size, hidden_layer_sizes=hidden_layer_sizes) # call `build_all_models` for the given hyperparameters

# search for the optimal combination of parameters given this criterion
max_score = 0
best_params = None
for a, act, b, h in res:
    if res[(a, act, b, h)]['val'] > max_score:
        max_score = res[(a, act, b, h)]['val']
        best_params = (a, act, b, h)

print(f"Best parameters: {best_params}")
print(f"Best score: {max_score}")

C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.wa

Best parameters: (0.01, 'relu', 16, (50,))
Best score: 0.8377581120943953


C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [25]:
# with bag of words
# Best parameters: (0.0001, 'relu', 32, (100,))
# Best score: 0.8643067846607669
# without bag of words
# Best parameters: (0.01, 'relu', 16, (50,))
# Best score: 0.8377581120943953

In [23]:
best_model_bag = train_neural_network(X_train, t_train, alpha=0.0001, activation='relu', batch_size=32, hidden_layer_sizes=(100,))
best_model_non_bag = train_neural_network(X_train, t_train, alpha=0.01, activation='relu', batch_size=16, hidden_layer_sizes=(50,))

C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [29]:
print(metrics.classification_report(t_val, best_model_bag.predict(X_val)))

              precision    recall  f1-score   support

           0       0.80      0.91      0.85       113
           1       0.81      0.66      0.73       113
           2       0.80      0.83      0.82       113

    accuracy                           0.80       339
   macro avg       0.80      0.80      0.80       339
weighted avg       0.80      0.80      0.80       339



In [30]:
print(metrics.classification_report(t_val, best_model_non_bag.predict(X_val)))

              precision    recall  f1-score   support

           0       0.83      0.88      0.85       113
           1       0.81      0.67      0.73       113
           2       0.79      0.88      0.84       113

    accuracy                           0.81       339
   macro avg       0.81      0.81      0.81       339
weighted avg       0.81      0.81      0.81       339

